In [1]:
pip install transformers datasets peft accelerate scikit-learn

   ---------------------------------------- 0.0/526.6 kB ? eta -:--:--
   ---------------------------------------- 526.6/526.6 kB 9.0 MB/s  0:00:00
   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   ---------------------------------------- 557.0/557.0 kB 9.5 MB/s  0:00:00

  Attempting uninstall: dill

    Found existing installation: dill 0.4.0

    Uninstalling dill-0.4.0:

      Successfully uninstalled dill-0.4.0

   ---------------------------------------- 0/5 [dill]
   ---------------------------------------- 0/5 [dill]
   ---------------------------------------- 0/5 [dill]
   -------- ------------------------------- 1/5 [multiprocess]
   -------- ------------------------------- 1/5 [multiprocess]
   -------- ------------------------------- 1/5 [multiprocess]
   ---------------- ----------------------- 2/5 [datasets]
   ---------------- ----------------------- 2/5 [datasets]
   ---------------- ----------------------- 2/5 [datasets]
   ---------------- ----

In [2]:
import pandas as pd

data = {
    "text": [
        "I forgot my password, please help me reset it",
        "This is urgent, reset my account immediately or I will complain",
        "Can you check my account balance?",
        "I am not the customer but I need access to this account",
        "Please update my contact details",
        "I lost my phone and need OTP bypass",
        "Just checking my last transactions",
        "Give me access to this account right now"
    ],
    "label": [0, 1, 0, 1, 0, 2, 0, 1]
}

df = pd.DataFrame(data)

In [3]:
# Step 3: Convert to HuggingFace Dataset
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

In [4]:
# Step 4: Tokenization
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
# Step 6: Apply LoRA (VERY IMPORTANT 🔥)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],  # DistilBERT attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [7]:
pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fraud_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
# Step 8: Trainer
from transformers import Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

In [11]:
# Step 9: Train Model
trainer.train()



c:\Users\admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=3, training_loss=1.0899464289347331, metrics={'train_runtime': 29.4118, 'train_samples_per_second': 0.612, 'train_steps_per_second': 0.102, 'total_flos': 2425394368512.0, 'train_loss': 1.0899464289347331, 'epoch': 3.0})

In [12]:
# ✅ Step 10: Save LoRA Adapter
model.save_pretrained("fraud_lora_adapter")
tokenizer.save_pretrained("fraud_lora_adapter")

('fraud_lora_adapter\\tokenizer_config.json',
 'fraud_lora_adapter\\tokenizer.json')

In [13]:
# ✅ Step 11: Inference
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

text = "Please reset my account immediately, I forgot everything"
result = classifier(text)

print(result)

[{'label': 'LABEL_0', 'score': 0.3660880923271179}]
